# Required Libraries and Functions

In [0]:
import glob, re, json
from pyspark.sql.functions import current_timestamp, col
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from calendar import monthrange

In [0]:
dbutils.widgets.text("volume_path", "/Volumes/workspace/taxi_weather/datasets/")

base_path = dbutils.widgets.get("volume_path")

# Determine Start and End Dates needed for the pipeline

In [0]:
# 1. Dynamically find all trip files
raw_trips_path = f"{base_path}yellow_tripdata_*.parquet"
trip_files = glob.glob(raw_trips_path)

# 2. Extract YYYY-MM from filenames and find Min/Max
months = [re.search(r'(\d{4}-\d{2})', f).group(1) for f in trip_files]
min_month = min(months)
max_month = max(months)

# 3. Calculate exact start
start_date = f"{min_month}-01"

# 4. Pad the max_month by exactly 1 month to catch trailing drop-offs
year, month = map(int, max_month.split('-'))

if month == 12:
    pad_month = 1
    pad_year = year + 1
else:
    pad_month = month + 1
    pad_year = year

# Get the last day of the padded month
last_day = monthrange(pad_year, pad_month)[1]

# Format strings with leading zeros (e.g., '2026-08-31')
end_date = f"{pad_year}-{pad_month:02d}-{last_day:02d}"

# 4. Save these parameters for the Weather API notebook to use
date_params = {"start_date": start_date, "end_date": end_date}
with open(f"{base_path}pipeline_dates.json", "w") as f:
    json.dump(date_params, f)

print(f"Pipeline dates set: Start={start_date}, End={end_date}")

# Read the available Trip Data

In [0]:
trips_df = (
    spark.read.format("parquet")
    .load(raw_trips_path)
    .withColumn("source_file_name", col("_metadata.file_path"))
    .withColumn("ingest_time", current_timestamp())
)

# Read Zone Lookup Data

In [0]:
zone_schema = StructType([
    StructField("LocationID", IntegerType(), False),
    StructField("Borough", StringType(), True),
    StructField("Zone", StringType(), True),
    StructField("service_zone", StringType(), True)
])

zone_df = (
    spark.read.format("csv")
    .option("header", True)
    .schema(zone_schema)
    .load(f"{base_path}taxi_zone_lookup.csv")
    .withColumn("source_file_name", col("_metadata.file_path"))
    .withColumn("ingest_time", current_timestamp())
    )

# Write to Delta Tables

In [0]:
trips_df.write.format("delta").mode("overwrite").saveAsTable("workspace.taxi_weather.bronze_trips")

zone_df.write.format("delta").mode("overwrite").saveAsTable("workspace.taxi_weather.bronze_zone")